# Sample selection


### **Data needed**
1. Daily prices for all NYSE/AMEX listed securities for 1974-1986 and 2012-2024, with market cap
2. Quarterly Earnings Per Share - Excludig Extraordinary Items, Price Close - Quarter End, Earnings Announcement Data (for 1982-1987 and 2020-2025)



### **Data sources**

Wharton Research Data Services. "WRDS" wrds.wharton.upenn.edu, accessed 2026-06-04.

##### **CRSP Daily**

*Query parameters*
- primaryexch (Primary Exchange (A:AMEX, B:BATS, I:IEX, N:NYSE, Q:NASDAQ, R:NYSE ARCA, X:Unknown))
- securitynm (Security Name)
- securitytype (Security Type)
- permco (PERMNO)
- ticker (Ticker)
- dlycap (Daily Capitalization)
- dlyret (Daily Total Return)

*Files*
- ```CRSP_Daily_1974_1986.csv``` (2.09GB)         From 1974-01-01 to 1986-12-31
- ``CRSP_Daily_2012_2025.csv`` (469.0MB)                  From 2012-01-01 to 2024-12-31


##### **Compustat Quarterly**

*Query parameters*
- conm (Company Name)
- tic (Ticker Symbol)
- rdq (Report Date of Quarterly Earnings)
- fyearq (Fiscal Year)
- fqtr (Fiscal Quarter)
- prccq (Price Close - Quarter)
- epsfxq (Earnings Per Share (Diluted) - Excluding Extraordinary items)
- epspxq (Earnings Per Share (Basic) - Excluding Extraordinary Items)
- cshoq (Common Shares Outstanding)
- cshprq (Common Shares Used to Calculate Earnings Per Share - Basic)
- ajexq (Adjustment Factor (Company) - Cumulative by Ex-Date)

*Files*
- ``Compustat_Q_196103_198712.csv`` (48.2MB)    From 1961-03 to 1987-12
- ``Compustat_Q_199903_202512.csv`` (137.2MB)   From 1999-03 to 2025-12

##### **Mapping table**

*Query parameters*
- gvkey
- conm
- tic
- cusip
- LINKPRIM
- LIID
- LINKTYPE
- LPERMNO
- LPERMCO
- LINKDT
- LINKENDDT

*Files*
- ``Compustat_CRSP_link.csv`` (3.3MB)

### **Paper methodology**

"*Our sample includes 84,792 firm-quarters of data for NYSE/AMEX firms for 1974-86. Criteria for inclusion in the sample are the same as those used by FOS, who studied NYSE/AMEX firms for the period 1974-81. We require that the firm be listed on the CRSP daily files, and that the firm's earnings before extraordinary items and discontinued operations be available for at least ten consecutive quarters on Compustat. Our NYSE/AMEX sample includes only firms that appeared on any of the Compustat files released from 1982 through 1987*."

### **Replication**
We first load Compustat Quarterly dataset obtained from WRDS.

In [17]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

dfQ = pd.read_csv("../data/WRDS/Compustat_Q_199903_202512.csv")
dfQ['datadate'] = pd.to_datetime(dfQ['datadate'])
dfQ.head()

,costat,curcdq,datafmt,indfmt,consol,tic,datadate,gvkey,conm,ajexq,fqtr,fyearq,rdq,cshoq,cshprq,epsfxq,epspxq,prccq
0,I,USD,STD,INDL,C,4165A,1999-03-31,1010,ACF INDUSTRIES INC,1.000,1,1999,NaN,0.015,NaN,NaN,NaN,NaN
1,I,USD,STD,INDL,C,AFAP,1999-03-31,1019,AFA PROTECTIVE SYSTEMS INC,1.000,1,1999,NaN,NaN,0.172,1.52,1.52,206.000
2,I,USD,STD,INDL,C,IWKS,1999-03-31,1021,AFP IMAGING CORP,0.002,3,1999,NaN,9.271,9.271,-0.13,-0.13,0.313
3,I,USD,STD,INDL,C,ALO.2,1999-03-31,1034,ALPHARMA INC -CL A,1.000,1,1999,1999-04-28,27.423,27.255,0.27,0.27,39.250
4,I,USD,STD,INDL,C,UDI.,1999-03-31,1036,UNITED DOMINION INDUSTRIES,1.000,1,1999,1999-04-19,NaN,40.331,0.33,0.33,19.875


Following Foster's (1977) methodology used by Bernard & Thomas (1989), we only shortlist firms that are present in the 1982-1987 Compustat files.

In [18]:
dfQ_slice = dfQ[(dfQ['datadate'] >= '2020-01-01') & (dfQ['datadate'] <= '2025-12-31')]
tickers_2020_2025 = dfQ_slice['tic'].dropna().unique()
print(f"Number of unique tickers in the 2020-2025 window: {len(tickers_2020_2025)}")

Number of unique tickers in the 2020-2025 window: 14819


Among these, we only keep firms that have at least ten consecutive earnings observations. We don't know if we should use epspxq (basic Earning per Share) instead of epsfxq (diluted EPS).


In [ ]:
# Checking for 10 consecutive quarterly announcement for shortlisted firms
dfQ = dfQ.sort_values(by=["tic", "fyearq", "fqtr"])
dfQ["pidx"] = dfQ["fyearq"] * 4 + dfQ["fqtr"]

df_filtered = dfQ[dfQ["tic"].isin(tickers_2020_2025)].copy()
sub = df_filtered[df_filtered["epsfxq"].notna()].copy() # epsfxq
sub["break"] = sub.groupby("tic")["pidx"].diff().ne(1)
sub["block"] = sub.groupby("tic")["break"].cumsum()

# 10 or more consecutive quarters
sizes = sub.groupby(["tic", "block"]).size().reset_index(name="count")
valid_firms = sizes[sizes["count"] >= 10]["tic"].unique()
print(f"{len(valid_firms)} shortlisted firms (10-consecutive quarters, ticker present in 2020-2025)")
dfQ.head()

# Shortlisted Compustat DF, without veryfing exchange or CRSP presence
valid_firms = dfQ[dfQ['tic'].isin(valid_firms)][['tic', 'gvkey']].drop_duplicates(subset=['tic'])

7452 shortlisted firms (10-consecutive quarters, ticker present in 2020-2025)


We then load CRSP Daily files, filtering firms not listed on NYSE/AMEX, following the authors method.

In [20]:
# Loading data
dfD_NYSE = pd.read_csv("../data/WRDS/CRSP_Daily_2012_2025.csv")

# We only select NYSE/AMEX listed firms
dfD_NYSE = dfD_NYSE[dfD_NYSE['PrimaryExch'].isin(['A', 'N'])]

print(f"CRSD Daily observations (2012-2025): {len(dfD_NYSE)}\n   {len(dfD_NYSE['PERMNO'].unique())} firms (PERMNO)")
dfD_NYSE.head()

CRSD Daily observations (2012-2025): 9193068
   4939 firms (PERMNO)


,PERMNO,HdrCUSIP,PrimaryExch,SecurityNm,SecurityType,Ticker,PERMCO,DlyCalDt,DlyCap,DlyRet,DlyClose,DlyLow,DlyHigh,DlyOpen
0,10001,36720410,A,GAS NATURAL INC; COM NONE; CONS,EQTY,EGAS,7953,2012-01-03,93281.76,0.001751,11.4400,11.3000,11.500,11.44
1,10001,36720410,A,GAS NATURAL INC; COM NONE; CONS,EQTY,EGAS,7953,2012-01-04,92303.28,-0.010490,11.3200,11.2500,11.450,11.38
2,10001,36720410,A,GAS NATURAL INC; COM NONE; CONS,EQTY,EGAS,7953,2012-01-05,92628.62,0.003525,11.3599,11.3199,11.360,11.35
3,10001,36720410,A,GAS NATURAL INC; COM NONE; CONS,EQTY,EGAS,7953,2012-01-06,92384.82,-0.002632,11.3300,11.2000,11.370,11.37
4,10001,36720410,A,GAS NATURAL INC; COM NONE; CONS,EQTY,EGAS,7953,2012-01-09,92303.28,-0.000883,11.3200,11.2680,11.332,11.28


Compustat firms are identifyed with their unique *gvkey*. CRSP firms are identified with their *PERMNO*. We use the CRSP/Compustat linking table to match both shortlisted firms from Compustat to shortlisted firms present in CRSP Daily files.

In [ ]:
mapping = pd.read_csv("../data/WRDS/Compustat_CRSP_link.csv")

# Filtering link type
mapping_filtered = mapping[(mapping['LINKTYPE'].isin(['LC', 'LU'])) & (mapping['LINKPRIM'].isin(['P', 'C']))].copy()

# Time filter on links
mapping_filtered['LINKDT']  = pd.to_datetime(mapping_filtered['LINKDT'],  errors='coerce')
mapping_filtered['LINKENDDT'] = pd.to_datetime(mapping_filtered['LINKENDDT'], errors='coerce')

mapping_filtered = mapping_filtered[(mapping_filtered['LINKDT'].isna()  | (mapping_filtered['LINKDT']  <= pd.Timestamp('2025-12-31'))) & (mapping_filtered['LINKENDDT'].isna() | (mapping_filtered['LINKENDDT'] >= pd.Timestamp('2020-01-01'))) ]

mapping_filtered['gvkey'] = mapping_filtered['gvkey'].astype(str)
valid_firms['gvkey']      = valid_firms['gvkey'].astype(str)

# Joining gvkey Compustat with shortlisted PERMNO
matched_ids = pd.merge(
    valid_firms[['gvkey']],
    mapping_filtered[['gvkey', 'LPERMNO', 'LINKDT', 'LINKENDDT']],
    on='gvkey',
    how='inner'
).rename(columns={'LPERMNO': 'PERMNO'})

# Matching with NYSE/AMEX
crsp_permnos = dfD_NYSE['PERMNO'].unique()
final_ids = matched_ids[matched_ids['PERMNO'].isin(crsp_permnos)].copy()

# Handling duplicate: we keep those covering the largest period
final_ids = (
    final_ids
    .sort_values('LINKENDDT', ascending=False, na_position='first')
    .drop_duplicates(subset='gvkey', keep='first')
    .drop_duplicates(subset='PERMNO', keep='first')
    [['gvkey', 'PERMNO']]
)

print(f"{len(final_ids)} unique gvkey-PERMNO pairs")


2688 unique gvkey-PERMNO pairs


These 2732 firms are our final datasets of interest: we applied all filters used by the researchers.

In [ ]:
# Clean dataframes
dfQ['gvkey'] = dfQ['gvkey'].astype(str)
dfQ_final = dfQ.merge(final_ids, on='gvkey', how='inner')[
    ['gvkey', 'PERMNO', 'tic', 'datadate', 'fqtr', 'fyearq', 'epspxq', 'prccq', 'rdq']
]

dfD_NYSE = dfD_NYSE.sort_values(['PERMNO', 'DlyCalDt'])


dfD_final = dfD_NYSE.merge(final_ids, on='PERMNO', how='inner')[
    ['gvkey', 'PERMNO', 'Ticker', 'DlyCalDt', 'DlyRet', 'DlyCap']
]

dfQ_final = dfQ_final[(dfQ_final['datadate'] >= '2012-01-01') & (dfQ_final['datadate'] <= '2024-12-31')]
dfQ_final = dfQ_final[dfQ_final['epspxq'].notna()]

# Added this
dfQ_final = dfQ_final.sort_values('rdq', ascending=False).drop_duplicates(
    subset=['gvkey', 'fyearq', 'fqtr'], keep='first'
)
# also deduplicate on datadate directly since that's your merge key later
dfQ_final = dfQ_final.drop_duplicates(subset=['gvkey', 'datadate'], keep='first')

print(f"{len(dfQ_final)} firm-quarters")

dfD_NYSE.to_csv("../data/OOS/dfD_NYSE.csv", index=False)
dfQ_final.to_csv("../data/OOS/dfQ_final.csv", index=False)
dfD_final.to_csv("../data/OOS/dfD_final.csv", index=False)

111859 firm-quarters


We find a total of 114628 firm-quarters. This is above the 84792 obtained in the paper, but we haven't fully cleaned them yet.